Bottom line:  
LangChain Documents are better for RAG because:  

They maintain the same structure throughout (load → split → embed → retrieve)  
They work seamlessly with LangChain's retrieval chains  
Metadata travels automatically with the content  

The confusion in my earlier example was mixing manual storage syntax with LangChain concepts. If you're using LangChain for RAG, stick with Document objects throughout. If you're building everything custom, you can use dicts but you lose the convenience.  

Let's do LangChain concetps, forget manual storage by now. I like Recommended approach for your use case. Observations:  
You import from langchain_openai import OpenAIEmbeddings and you don't use it  
Could you please retrieve the source to the chunk metadata?  
I will use chromadb with persistance.  
Could you  fetch me some code?  

In [ ]:
import json
import numpy as np
from pathlib import Path
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document

# 1. Load your chunks (from earlier in the pipeline)
# Assuming you have `chunks` list from the splitting step

# 2. Get batch results and match with chunks
def process_batch_results(batch_results, chunks):
    """
    Match batch embedding results with original chunks
    
    Args:
        batch_results: List of batch API results
        chunks: List of LangChain Document objects
    
    Returns:
        Dict mapping custom_id to embedding and document
    """
    embeddings_dict = {}
    
    for result in batch_results:
        custom_id = result['custom_id']
        embedding = result['response']['body']['data'][0]['embedding']
        
        # Parse custom_id to find matching chunk
        docid, start_index = custom_id.split("_")
        chunk = next(
            c for c in chunks 
            if c.metadata['docid'] == int(docid) 
            and c.metadata['start_index'] == int(start_index)
        )
        
        embeddings_dict[custom_id] = {
            'embedding': embedding,
            'document': chunk
        }
    
    return embeddings_dict

# 3. Create ChromaDB with pre-computed embeddings
def create_chroma_vectorstore(embeddings_dict, persist_directory="./chroma_db"):
    """
    Create ChromaDB vector store from pre-computed embeddings
    
    Args:
        embeddings_dict: Dict from process_batch_results
        persist_directory: Where to persist the database
    
    Returns:
        Chroma vector store
    """
    # Prepare data
    documents = []
    embeddings = []
    ids = []
    
    for custom_id, item in embeddings_dict.items():
        documents.append(item['document'])
        embeddings.append(item['embedding'])
        ids.append(custom_id)
    
    # Create Chroma collection with pre-computed embeddings
    vectorstore = Chroma.from_documents(
        documents=documents,
        embedding=None,  # We'll add embeddings manually
        persist_directory=persist_directory,
        ids=ids
    )
    
    # Note: Chroma.from_documents will compute embeddings by default
    # To use pre-computed embeddings, we need a different approach:
    
    return vectorstore

# Better approach: Add documents with embeddings directly
def create_chroma_with_precomputed_embeddings(embeddings_dict, persist_directory="./chroma_db"):
    """
    Create ChromaDB with pre-computed embeddings
    """
    import chromadb
    from chromadb.config import Settings
    
    # Initialize ChromaDB client
    client = chromadb.PersistentClient(path=persist_directory)
    
    # Create or get collection
    collection = client.get_or_create_collection(
        name="financial_filings",
        metadata={"description": "6-K financial filings embeddings"}
    )
    
    # Prepare data
    ids = []
    embeddings = []
    documents = []
    metadatas = []
    
    for custom_id, item in embeddings_dict.items():
        ids.append(custom_id)
        embeddings.append(item['embedding'])
        documents.append(item['document'].page_content)
        
        # Add source to metadata
        metadata = item['document'].metadata.copy()
        # Extract filename from source path
        source_path = Path(metadata.get('source', ''))
        metadata['source'] = source_path.name  # Just the filename
        metadata['source_path'] = str(source_path)  # Full path
        
        metadatas.append(metadata)
    
    # Add to collection
    collection.add(
        ids=ids,
        embeddings=embeddings,
        documents=documents,
        metadatas=metadatas
    )
    
    print(f"Added {len(ids)} documents to ChromaDB")
    print(f"Persisted to: {persist_directory}")
    
    return collection

# 4. Query the vector store
def similarity_search(collection, query_text, n_results=5):
    """
    Perform similarity search
    
    Args:
        collection: ChromaDB collection
        query_text: Query string
        n_results: Number of results to return
    
    Returns:
        Search results with documents and metadata
    """
    from openai import OpenAI
    
    # Get query embedding
    client = OpenAI()
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=query_text
    )
    query_embedding = response.data[0].embedding
    
    # Search
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results
    )
    
    return results

# 5. Load existing vector store
def load_chroma_vectorstore(persist_directory="./chroma_db"):
    """
    Load existing ChromaDB vector store
    """
    import chromadb
    
    client = chromadb.PersistentClient(path=persist_directory)
    collection = client.get_collection(name="financial_filings")
    
    print(f"Loaded collection with {collection.count()} documents")
    
    return collection

# Example usage:
if __name__ == "__main__":
    # Assuming you have batch_results and chunks from earlier steps
    
    # Process batch results
    embeddings_dict = process_batch_results(batch_results, chunks)
    
    # Create and persist vector store
    collection = create_chroma_with_precomputed_embeddings(
        embeddings_dict, 
        persist_directory="./chroma_db"
    )
    
    # Later, load and query
    collection = load_chroma_vectorstore("./chroma_db")
    
    results = similarity_search(
        collection, 
        "What was the passenger traffic in October 2025?",
        n_results=5
    )
    
    # Print results
    print("\nSearch Results:")
    for i, (doc, metadata, distance) in enumerate(zip(
        results['documents'][0],
        results['metadatas'][0], 
        results['distances'][0]
    )):
        print(f"\n--- Result {i+1} ---")
        print(f"Source: {metadata['source']}")
        print(f"Distance: {distance:.4f}")
        print(f"Content: {doc[:200]}...")  # First 200 chars

Key points:  

✅ No unused imports - Removed OpenAIEmbeddings since we use pre-computed embeddings  
✅ Source in metadata - Extracts filename from path and adds both source (filename) and source_path (full path)  
✅ ChromaDB with persistence - Uses PersistentClient to save to disk  
✅ Pre-computed embeddings - Adds your batch embeddings directly without recomputing  
✅ Easy retrieval - Load collection and search anytime  

The database persists in ./chroma_db directory and can be loaded later without recomputing embeddings!  